Leitura dos arquivos CSV da camada Bronze

In [0]:
df_uber_a = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/uber/vianorte/*.csv",
    header=True,
    inferSchema=True,
    sep=";"
)


In [0]:

df_uber_b = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/uber/atlas/*.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

Validação da estrutura dos dados

In [0]:
display(df_uber_a)
display(df_uber_b)

df_uber_a.printSchema()
df_uber_b.printSchema()

print(df_uber_a.columns)
print(df_uber_b.columns)

Padronização das colunas das agências

In [0]:
from pyspark.sql.functions import col, lit

df_uber_a_padrao = df_uber_a.select(
    col("Corrida ID").alias("id_corrida"),
    col("Colaborador").alias("viajante"),
    col("Data Corrida").alias("data_corrida"),
    col("Centro de Custo").alias("centro_custo_origem"),
    col("Origem").alias("origem"),
    col("Destino").alias("destino"),
    col("Valor Corrida").alias("valor")
).withColumn(
    "agencia",
    lit("ViaNorte")
)

In [0]:
from pyspark.sql.functions import col, lit

df_uber_b_padrao = df_uber_b.select(
    col("TripCode").alias("id_corrida"),
    col("Usuario").alias("viajante"),
    col("Data").alias("data_corrida"),
    col("CCusto").alias("centro_custo_origem"),
    col("Ponto Inicial").alias("origem"),
    col("Ponto Final").alias("destino"),
    col("Total").alias("valor")
).withColumn(
    "agencia",
    lit("Atlas Corporate")
)

In [0]:
df_uber_a_padrao.columns == df_uber_b_padrao.columns

Limpeza de registros vazios

In [0]:
df_uber_a_padrao = df_uber_a_padrao.filter(
    col("id_corrida").isNotNull()
)

df_uber_b_padrao = df_uber_b_padrao.filter(
    col("id_corrida").isNotNull()
)

In [0]:
print("Agência A:", df_uber_a_padrao.count())
print("Agência B:", df_uber_b_padrao.count())

União das bases padronizadas

In [0]:
df_uber = df_uber_a_padrao.unionByName(
    df_uber_b_padrao
)

In [0]:
print("Total de corridas:",df_uber.count())

In [0]:
display(df_uber)

Tratamento do valor monetário

In [0]:
from pyspark.sql.functions import regexp_replace

In [0]:
df_uber = df_uber.withColumn(
    "valor_numerico",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("valor"),
                "R\\$",
                ""
            ),
            "\\.",
            ""
        ),
        ",",
        "."
    ).cast("decimal(18,2)")
)

In [0]:
df_uber.select(
    "valor",
    "valor_numerico"
).show(20, truncate=False)

In [0]:
df_uber.printSchema()

Criação de colunas analíticas

In [0]:
from pyspark.sql.functions import lit, month, when

In [0]:
df_uber = df_uber.withColumn(
    "tipo_despesa",
    lit("Uber")
)

In [0]:
df_uber = df_uber.withColumn(
    "trimestre",
    when(
        month(col("data_corrida")).between(1,3),
        "Primeiro Trimestre"
    )
    .when(
        month(col("data_corrida")).between(4,6),
        "Segundo Trimestre"
    )
    .when(
        month(col("data_corrida")).between(7,9),
        "Terceiro Trimestre"
    )
    .otherwise(
        "Quarto Trimestre"
    )
)

In [0]:
df_uber.select(
    "data_corrida",
    "tipo_despesa",
    "trimestre"
).show(30, truncate=False)

In [0]:
df_uber.groupBy("trimestre").count().show()

Padronização do centro de custo

In [0]:
df_map = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/referencias/centro_custo/Centro_Custo_Map.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

In [0]:
df_uber = (
    df_uber.alias("u")
    .join(
        df_map.alias("m"),
        (
            (col("u.agencia") == col("m.Agencia")) &
            (col("u.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "u.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_uber.select(
    "agencia",
    "centro_custo_origem",
    "centro_custo",
    "diretoria"
).show(30, truncate=False)

In [0]:
print("Total de corridas:", df_uber.count())

print(
    "Centros de custo não mapeados:",
    df_uber.filter(
        col("centro_custo").isNull()
    ).count()
)

Persistência da camada Silver

In [0]:
df_uber.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.silver.uber")

In [0]:
df_uber_teste = spark.table(
    "viagens_teste.silver.uber"
)

display(df_uber_teste)

In [0]:
print("Total de corridas:", df_uber.count())

print(
    "Centros de custo não mapeados:",
    df_uber.filter(
        col("centro_custo").isNull()
    ).count()
)